# Кластеризация: поиск естественных групп в данных

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Кластеризация».

## Что делает метод

Кластеризация отвечает на вопрос «есть ли в моих данных естественные группы». Заранее заданных меток нет: алгоритм сам объединяет похожие наблюдения и разделяет непохожие. Это инструмент разведочного этапа исследования — увидеть, распадается ли выборка на типы клеток, режимы процесса, профили пациентов или сегменты респондентов.

В этом блокноте мы применим два метода — K-средних и иерархическую кластеризацию, — оценим качество разбиения и разберём, как читать результат. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы рассыпали на столе разноцветные предметы: скрепки, монеты, пуговицы, карандаши. Не получив никакого списка категорий заранее, вы интуитивно начинаете группировать похожее с похожим. Алгоритм кластеризации делает то же самое с числовыми данными: он измеряет «расстояние» между наблюдениями и объединяет близкие в группы.

Принципиальное отличие от классификации: правильных ответов нет. Метод сам обнаруживает структуру — поэтому его используют в самом начале исследования, когда ещё неизвестно, на какие группы делятся данные.

**Ключевые понятия, которые встретятся в блокноте:**

- **Признак** — один измеренный параметр наблюдения (столбец таблицы).
- **Стандартизация признаков** — приведение всех признаков к единому масштабу (среднее 0, разброс 1), чтобы признаки с большими числовыми значениями не доминировали над остальными.
- **Кластер** — группа наблюдений, похожих друг на друга и непохожих на наблюдения из других групп.
- **Метрика качества кластеризации** — числовая оценка того, насколько хорошо кластеры разделены.
- **Коэффициент силуэта** — метрика от -1 до 1: чем выше, тем плотнее кластеры и чётче граница между ними.
- **Главная компонента** — новая ось, построенная алгоритмом PCA так, чтобы вдоль неё данные разбросаны максимально широко; используется для отображения многомерных данных на плоскости.
- **Дендрограмма** — дерево слияний иерархической кластеризации; высота ветвления показывает, насколько различны объединяемые группы.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2 scipy==1.13.1

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор по геометрическим характеристикам зёрен пшеницы трёх сортов (`seeds`, по умолчанию загружается с публичного репозитория UCI). Метки сорта в кластеризации не используются — они нужны лишь в конце, чтобы оценить, насколько найденные группы соответствуют реальным сортам. При недоступности сети ячейка формирует эквивалентный синтетический набор.

In [ ]:
import numpy as np
import pandas as pd

columns = ["area", "perimeter", "compactness", "kernel_length",
           "kernel_width", "asymmetry", "groove_length", "variety"]
url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "00236/seeds_dataset.txt")
try:
    df = pd.read_csv(url, sep=r"\s+", names=columns)
    print("Набор seeds загружен из репозитория UCI.")
except Exception as err:
    print("Сеть недоступна, формируем синтетический аналог:", err)
    from sklearn.datasets import make_blobs
    Xb, yb = make_blobs(n_samples=210, n_features=7, centers=3,
                        cluster_std=1.4, random_state=42)
    df = pd.DataFrame(Xb, columns=columns[:-1])
    df["variety"] = yb + 1

X = df.drop(columns=["variety"])
true_labels = df["variety"]
print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
X.head()

## 4. Применение метода

**Шаг 4.1. Стандартизация признаков.**

Признаки в этом наборе имеют разные единицы измерения и разный числовой масштаб. Без стандартизации расстояние между наблюдениями будет определяться главным образом признаком с наибольшим разбросом — остальные окажутся «немыми». `StandardScaler` приводит каждый признак к нулевому среднему и единичному стандартному отклонению.

**Шаг 4.2. Выбор числа кластеров по коэффициенту силуэта.**

K-средних требует заранее указать число кластеров `k`. Чтобы выбрать его осмысленно, перебираем `k` от 2 до 7 и для каждого вычисляем **коэффициент силуэта** — меру того, насколько наблюдения внутри кластера ближе к своим «соседям», чем к наблюдениям из ближайшего другого кластера. Значение от -1 до 1: выше — лучше.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X)

candidates = range(2, 8)
silhouettes = []
for k in candidates:
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

best_k = list(candidates)[int(np.argmax(silhouettes))]
print(f"Лучшее число кластеров по силуэту: {best_k}")

**Шаг 4.3. Финальное разбиение и оценка согласованности.**

Применяем K-средних с выбранным числом кластеров. Затем оцениваем **ARI (скорректированный индекс Ранда)** — насколько найденные кластеры совпадают с известными метками сортов. ARI = 1 означает идеальное совпадение, ARI = 0 — совпадение на уровне случайного. Важно: в реальной кластеризации известных меток нет; здесь они используются только для проверки.

In [ ]:
# Финальное разбиение методом K-средних с выбранным числом кластеров.
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, cluster_labels)
print(f"Согласованность с известными сортами (ARI): {ari:.3f}")

### Шаг 4.4. Визуализация результата

Ячейка ниже строит два графика. Левый — качество разбиения по числу кластеров. Правый — найденные кластеры на двумерной карте главных компонент. Поскольку данные семимерные, отобразить их напрямую невозможно — PCA проецирует их на плоскость двух главных осей, сохраняя максимум информации.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Силуэт по числу кластеров
axes[0].plot(list(candidates), silhouettes, marker="o", color=VIZ["series"][0])
axes[0].axvline(best_k, color=VIZ["series"][2], linestyle="--",
                label=f"Выбрано: {best_k}")
axes[0].set_title("Качество разбиения по числу кластеров")
axes[0].set_xlabel("Число кластеров")
axes[0].set_ylabel("Коэффициент силуэта")
axes[0].legend(loc="best")

# Кластеры на плоскости главных компонент
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
palette = get_palette(best_k)
for c in range(best_k):
    mask = cluster_labels == c
    axes[1].scatter(coords[mask, 0], coords[mask, 1], s=28, alpha=0.8,
                    color=palette[c], label=f"Кластер {c + 1}")
axes[1].set_title("Найденные кластеры (проекция PCA)")
axes[1].set_xlabel("Главная компонента 1")
axes[1].set_ylabel("Главная компонента 2")
axes[1].legend(loc="best")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Силуэт по числу кластеров (левый)**: кривая показывает качество разбиения при разных `k`. Пик кривой — кандидат на оптимальное число групп. Пунктирная линия отмечает выбранное значение. Обратите внимание: пик может не совпасть с реальным числом групп, если данные не имеют чёткой кластерной структуры.

- **Карта кластеров PCA (правый)**: каждая точка — одно наблюдение, цвет — найденный кластер. Компактные, не перекрывающиеся облака означают чёткое разбиение. Перекрытие зон говорит о том, что граница между группами условна — или что двух главных компонент недостаточно для полного отображения структуры.

### Шаг 4.5. Иерархическая кластеризация

Иерархическая кластеризация строит дерево вложенности — **дендрограмму** — и не требует заранее задавать число групп. Метод Ward минимизирует дисперсию внутри кластеров при каждом слиянии. Ячейка ниже строит усечённую дендрограмму (последние 20 слияний), чтобы структура была читаемой.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

linkage_matrix = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(12, 5.2))
dendrogram(linkage_matrix, truncate_mode="lastp", p=20,
           color_threshold=0, ax=ax,
           link_color_func=lambda _: VIZ["edge"])
ax.set_title("Дендрограмма иерархической кластеризации")
ax.set_xlabel("Наблюдения (свёрнутые ветви)")
ax.set_ylabel("Расстояние объединения")
ax.grid(True, axis="y")
fig.tight_layout()
plt.show()

**Как читать дендрограмму:**

Вертикальная ось — расстояние объединения: чем выше точка слияния двух ветвей, тем более различны объединяемые группы. Читайте дерево снизу вверх: внизу объединяются самые похожие наблюдения, наверху — самые разные.

Чтобы выбрать число кластеров, проведите мысленную горизонтальную линию там, где вертикальные отрезки длиннее всего (наибольший «прыжок» по высоте). Число ветвей, которые пересекает эта линия — рекомендуемое число кластеров.

## 5. Интерпретация результата

- **Коэффициент силуэта** меняется от -1 до 1; значения ближе к 1 означают плотные, хорошо разделённые кластеры. Пик кривой подсказывает разумное число групп, но окончательное решение опирается и на содержательную интерпретируемость.
- **Карта кластеров (PCA)**: компактные, не перекрывающиеся группы указывают на устойчивое разбиение. Сильное перекрытие означает, что граница между группами условна.
- **ARI** показывает согласованность с известной разметкой (когда она доступна для проверки); значение около 1 — почти полное совпадение, около 0 — совпадение на уровне случайного.
- **Дендрограмма**: длинные вертикальные отрезки соответствуют объединению заметно различных кластеров — это естественные места для «разреза» дерева.

Помните: алгоритм всегда вернёт разбиение, даже если естественной кластерной структуры в данных нет; низкий силуэт и сильное перекрытие на карте — повод усомниться в наличии групп.

## 5б. Наглядный эксперимент: кластеризация «с нуля» на синтетических данных

Создадим простой двумерный пример с четырьмя явными группами. Покажем сразу: данные без разметки, результат K-средних и центроиды кластеров. Так видно, что алгоритм «находит» без каких-либо подсказок.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Синтетические данные: четыре компактных облака.
X_demo, _ = make_blobs(n_samples=400, centers=[[-5, 4], [5, 4], [-5, -4], [5, -4]],
                        cluster_std=1.2, random_state=0)
X_demo_s = StandardScaler().fit_transform(X_demo)

km_demo = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_demo = km_demo.fit_predict(X_demo_s)
centers_demo = km_demo.cluster_centers_

palette = get_palette(4)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# Левый: данные без разметки.
axes[0].scatter(X_demo_s[:, 0], X_demo_s[:, 1], s=20, alpha=0.5,
                color=VIZ["edge"])
axes[0].set_title("Данные до кластеризации\n(разметки нет)")
axes[0].set_xlabel("Признак 1 (стандартизованный)")
axes[0].set_ylabel("Признак 2 (стандартизованный)")

# Правый: найденные кластеры и центроиды.
for c in range(4):
    mask = labels_demo == c
    axes[1].scatter(X_demo_s[mask, 0], X_demo_s[mask, 1],
                    s=20, alpha=0.6, color=palette[c], label=f"Кластер {c+1}")
axes[1].scatter(centers_demo[:, 0], centers_demo[:, 1],
                s=180, color="white", edgecolors=VIZ["node_text"],
                linewidths=2, zorder=5, marker="*", label="Центроид")
axes[1].set_title("Результат K-средних (k=4)")
axes[1].set_xlabel("Признак 1 (стандартизованный)")
axes[1].set_ylabel("Признак 2 (стандартизованный)")
axes[1].legend(loc="upper right", fontsize=9)

fig.tight_layout()
plt.show()

**Как читать этот график:**

Левый — «сырые» данные: точки без цвета, структура неочевидна. Правый — результат: алгоритм, не зная ничего о природе данных, разделил их на четыре группы, которые визуально соответствуют реальным облакам. Звёздочки — **центроиды** кластеров: усреднённое положение всех точек группы.

Обратите внимание: идеального разбиения нет — несколько точек на границах между облаками могут попасть «не в свой» кластер. Это нормально: алгоритм оптимизирует суммарное расстояние до центроидов, а не истинную принадлежность.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу числовых признаков; столбец с метками не нужен — кластеризация работает без них.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и при необходимости укажите столбцы, не являющиеся признаками.
3. Последовательно выполните ячейки разделов 4 и 5.

## Поэкспериментируйте сами

На синтетическом примере (раздел 5б) попробуйте:

| Что изменить | Что наблюдать |
|---|---|
| `n_clusters=2` вместо 4 | Как алгоритм «сольёт» соседние облака в одно? |
| `cluster_std=3.0` в `make_blobs` | Что произойдёт с силуэтом при перекрывающихся облаках? |
| Убрать стандартизацию (`X_demo_s = X_demo`) | Изменится ли разбиение и почему? |

На основном наборе (раздел 4) попробуйте `candidates = range(2, 10)` — посмотрите, не появится ли второй пик силуэта при большем числе кластеров.

## Краткие выводы

- Кластеризация — метод обучения без учителя: правильных ответов нет, алгоритм сам обнаруживает структуру. Используйте его на разведочном этапе исследования.
- Стандартизация признаков перед кластеризацией обязательна: без неё расстояния неинформативны.
- Силуэт помогает выбрать число кластеров, но окончательное решение должно опираться на содержательную интерпретируемость групп.
- Алгоритм всегда вернёт разбиение, даже если структуры в данных нет: низкий силуэт и сильное перекрытие на карте PCA — сигнал отсутствия естественных групп.
- Иерархическая кластеризация не требует задавать число групп заранее и полезна для первичного анализа.
- Для отображения многомерных данных на плоскости используется PCA — только для визуализации, не для самой кластеризации.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На карте PCA два кластера практически слились, тогда как на карте t-SNE они разделены отчётливо. Какой из двух результатов следует считать более достоверным и что нужно сделать, чтобы убедиться в надёжности разделения?
2. Вы применили K-средних к своим данным без предварительной стандартизации, и коэффициент силуэта оказался высоким. Тем не менее кластеризация выглядит содержательно бессмысленной. В чём наиболее вероятная причина?
3. Алгоритм K-средних с `k=3` даёт силуэт 0.42, а с `k=5` — 0.45. Следует ли автоматически выбрать `k=5`? Какие дополнительные соображения необходимы для обоснованного выбора числа кластеров?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Ни один из результатов не является окончательным сам по себе. t-SNE сохраняет локальное соседство и часто разделяет группы нагляднее, однако расстояния между кластерами и их размеры не несут количественного смысла. Надёжный вывод требует: проверить устойчивость разбиения при изменении `perplexity`, запустить K-средних и иерархическую кластеризацию и сравнить результаты, оценить ARI или силуэт численно.
2. Без стандартизации признак с наибольшим числовым масштабом (например, площадь в квадратных метрах против бинарного флага) полностью доминирует при вычислении евклидовых расстояний. Высокий силуэт лишь отражает разделение по этому одному признаку, игнорируя все остальные. Необходимо применить `StandardScaler` и повторить кластеризацию.
3. Нет, разница в силуэте 0.03 может быть статистически незначимой. Окончательный выбор числа кластеров должен опираться на содержательную интерпретируемость групп (объясняют ли они что-то в предметной области?), устойчивость разбиения при разных случайных инициализациях, результат иерархической кластеризации (дендрограмма) и предметные ограничения задачи.
</details>


In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_data.csv")               # ваш файл
# id_columns = []                               # столбцы-идентификаторы, не признаки
#
# X = df.drop(columns=id_columns) if id_columns else df.copy()
# X = X.select_dtypes(include="number")         # оставляем только числовые признаки
# true_labels = pd.Series([0] * len(X))         # замените, если есть известная разметка
#
# print(f"Наблюдений: {X.shape[0]}, признаков: {X.shape[1]}")
# Далее повторите ячейки раздела 4.